<a href="https://colab.research.google.com/github/eyoung-15/IAT460_DJPal_Final_Project/blob/main/IAT_460_Final_Project_DJ_Pal_(April_5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#ensure pip is installed
!python3 -m pip --version

pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [ ]:
#install dependencies
!pip install librosa #for audio analysis and features
!pip install -U demucs # stem seperation
!pip install pydub #audio manipulation
!pip install soundfile #save and load audio files
!apt-get install -y ffmpeg #required for pydub

#Install for MusicGen
!python -m pip install 'torch==2.1.0' #install correct torch for MusicGen
!python -m pip install setuptools wheel #for MusicGen
!pip install git+https://github.com/huggingface/transformers.git #HuggingFace transformers


!pip install git+https://github.com/facebookresearch/demucs # demucs from github

!pip install gradio # Gradio used for UI

!pip install matplotlib #for spectrograms and plotting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.3/249.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 1.7 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=cdf86dda2f41c0a838ad8e56398eb441fd5fa630bdb5949cb51a79b079f656b8
  Stored in directory: /root/.cache/pip/wheels/1b/0c/20/a3b3daa1f9b65c8b0445729f94740ec335d0f86f1066c5c414
  Created wheel for julius: filename=julius-0.2.7-py

In [ ]:
import os #file handling
import numpy as np

import librosa
import soundfile as sf
import demucs
from pydub import AudioSegment #audio editing
from pydub.effects import normalize #normalize audio loudness
import torch
import torchaudio

#for Hybrid Demucs
from demucs.apply import apply_model
from demucs import separate
from demucs.pretrained import get_model

##MusicGen model
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import scipy

#for uploading and playing audio
from google.colab import files #allows file upload in google colab
from IPython.display import Audio, display #for audio playing in notebook

import matplotlib.pyplot as plt
import librosa.display

import gradio as gr
from scipy.signal import fftconvolve #used for reverb

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
#####DEVICE AND MODEL SETUP

#information on MusicGen technique: https://colab.research.google.com/github/sanchit-gandhi/notebooks/blob/main/MusicGen.ipynb

#Use GPU if available, otherwise use the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

#load MusicGen model (generates music from prompts)
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
musicgen_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)


#####GENRE PRESETS

#genre presets for controlling stem remix output
#gain = volume boost, reverb = echo effect strength, percussive boost = drum emphasis
Genre_Presets = {
    "EDM": {"gain": 1.5, "reverb": 0.15, "percussive_boost": 1.8},
    "Hip-Hop": {"gain": 2, "reverb": 0.05, "percussive_boost": 1.3},
    "Lo-Fi": {"gain": 1, "reverb": 0.4, "percussive_boost": 0.6},
    "Rock": {"gain": 2, "reverb": 0.1, "percussive_boost": 1.4},
    "Industrial": {"gain": 2.5, "reverb": 0.12, "percussive_boost": 1.2},
    "Punk": {"gain": 2, "reverb": 0.08, "percussive_boost": 1.6},
    "Pop": {"gain": 1.5, "reverb": 0.18, "percussive_boost": 1.1},
    "Metal": {"gain": 2.5, "reverb": 0.05, "percussive_boost": 1.5},
    "Cinematic": {"gain": 1.5, "reverb": 0.5, "percussive_boost": 0.5},
    "Grunge": {"gain": 2, "reverb": 0.25, "percussive_boost": 1.3},
    "Dubstep": {"gain": 3, "reverb": 0.08, "percussive_boost": 2.0},
    "Techno": {"gain": 2, "reverb": 0.1, "percussive_boost": 1.9},
}


#Music Gen prompts for each genre to generate optional backing track
def get_musicgen_prompt(genre):
    return {
        "EDM": "An energetic EDM track with heavy bass, fast tempo, and club beats",
        "Hip-Hop": "Hip hop beat with deep bass",
        "Lo-Fi": "A soft lo-fi chill beat with a mellow piano",
        "Rock": "Rock music with electric guitars and strong drums",
        "Industrial": "A dark industrial track with mechanical sounds and harsh synths",
        "Punk": "Fast, aggressive punk rock with electric guitars and drums",
        "Pop": "A catchy pop tune with vocals and bright synths",
        "Metal": "Heavy metal track with distorted guitars and drums",
        "Cinematic": "Cinematic orchestral score with dramatic build-up, emotional tension, and film soundtrack atmosphere",
        "Grunge": "Grunge track with distorted electric guitars, heavy drums, gritty tone, 90s alternative rock style, dark and aggressive mood",
        "Dubstep": "Dubstep electronic track with heavy bass drops, wobble bass, aggressive synths, fast tempo, and rhythmic build-ups and drops",
        "Techno": "Techno electronic track with kick drum, repetitive rhythm, deep bassline, minimal melodic elements, dark club atmosphere"

    }.get(genre, "An instrumental music track")


#target bpm for each genre
Target_Tempo = {
    "EDM": 130,
    "Hip-Hop": 90,
    "Lo-Fi": 75,
    "Rock": 110,
    "Industrial": 120,
    "Punk": 150,
    "Pop": 100,
    "Metal": 140,
    "Cinematic": 75,
    "Grunge": 95,
    "Dubstep": 140,
    "Techno": 130
}


#target musical key for each genre
Target_Keys = {
    "EDM": "C",
    "Hip-Hop": "A",
    "Lo-Fi": "D",
    "Rock": "E",
    "Industrial": "D#",
    "Punk": "G",
    "Pop": "C",
    "Metal": "E",
    "Cinematic": "D",
    "Grunge": "E",
    "Dubstep": "F",
    "Techno": "C#"
}


####AUDIO TRANSFORMATIONS

#convert uploaded songs to wav (wav required for processing)
def convert_to_wav(file_path):
  #extract file extension
  ext = os.path.splitext(file_path)[1].lower()

  #if already wav do not convert
  if ext == ".wav":
    return file_path


  audio = AudioSegment.from_file(file_path) #load audio file
  wav_path = os.path.splitext(file_path)[0] + ".wav" #create new file name with .wav
  audio.export(wav_path, format="wav") #convert and save audio as wav
  return wav_path  #path to converted file



#stem seperation (vocals,drums, bass, other) using demucs model
# Uses demucs.separate: https://github.com/facebookresearch/demucs/blob/main/demucs/separate.py
# gradio progress documentation: https://www.gradio.app/docs/gradio/progress
def separate_stems(audio_path, progress=gr.Progress()):
    progress(0.1, desc="Separating Stems...") #UI progress bar

    os.system(f'python -m demucs.separate -n htdemucs --out separated "{audio_path}"') #run Hybrid Demucs using command line

    #get most recent output folder
    model_folder = sorted(os.listdir("separated"))[-1]

    song_name = os.path.splitext(os.path.basename(audio_path))[0]  #extract song name from file path
    stem_folder = os.path.join("separated", model_folder, song_name)  #construct path to separated stems

    progress(0.4, desc="Stems Ready")

    #return dictionary of stem file paths
    return{
        "vocals": os.path.join(stem_folder, "vocals.wav"),
        "drums": os.path.join(stem_folder, "drums.wav"),
        "bass": os.path.join(stem_folder, "bass.wav"),
        "other": os.path.join(stem_folder, "other.wav")
    }


#######REMIXING

#apply genre eq function created by ChatGPT
def apply_eq(y, sr, genre):
    Y = np.fft.rfft(y) #convert audio from time domain to frequency domain
    freqs = np.fft.rfftfreq(len(y), 1/sr) #generate a array of frequencys corresponding to each FFT bin

    gain = np.ones_like(freqs) #create gain array initialized to 1 which will be modified

    if genre == "EDM":
        gain[freqs < 120] *= 1.8
        gain[freqs > 8000] *= 1.3

    #create warm tone and muffled vintage feel
    elif genre == "Lo-Fi":
        gain[freqs > 5000] *= 0.4
        gain[freqs < 200] *= 1.2

    ##enhance guitar and vocals
    elif genre == "Rock":
        gain[(freqs > 1000) & (freqs < 4000)] *= 1.5

    #wider boost range than rock
    elif genre == "Metal":
        gain[(freqs > 500) & (freqs < 5000)] *= 1.7
        gain[freqs < 100] *= 1.3

    #boost bass
    elif genre == "Hip-Hop":
        gain[freqs < 150] *= 1.6
        gain[freqs > 6000] *= 0.7

    Y *= gain #multiply each frequency component by its gain
    return np.fft.irfft(Y) #convert signal back to time domain


# distortion effect created by ChatGPT
def add_distortion(y, amount=2.0): #amount controls distortion intensity
    return np.tanh(amount * y)

#reverb function created by Google Gemini
def add_reverb(y, sr, strength=0.5, decay=0.15):

    ir_len = int(sr * decay) #length of impulse response based on sample rate and decay time
    #np.linspace generates an array of evenly spaced numbers over a specified interval
    #np.exp calculates exponential of all values in the array
    ir = np.exp(-np.linspace(0, 5, ir_len)) # Decaying curve (simulates natural reverb)
    ir /= ir.sum() # Normalize impulse response energy


    reverb = fftconvolve(y, ir, mode='full')[:len(y)] #applies convolution (adds echo effect)


    mixed = y + strength * reverb #mix original signal with reverb signal
    return np.clip(mixed, -1.0, 1.0) #prevent clipping (keep values in valid audio range)


def pitch_shifts(y, sr, n_steps, stem_type):
  #limit pitch shifts for vocals to avoid unnatural vocals
    if stem_type == "vocals":
      #np.clip limits the values in an array
        n_steps = np.clip(n_steps, -4, 4)
    return librosa.effects.pitch_shift(y, n_steps=n_steps, sr=sr)


#librosa remix stem function (applying genre specific remixing to a single stem)
def remix_stem(file_path, preset, genre, stem_name):
    y, sr = librosa.load(file_path, sr=None) #load audio and sample rate


    #tempo matching
    target_bpm = Target_Tempo[genre] #get the target tempo of the genre
    # beat_track documentation: https://librosa.org/doc/main/generated/librosa.beat.beat_track.html
    original_bpm, _ = librosa.beat.beat_track(y=y, sr=sr) #detect original tempo

    #for if detection fails
    if original_bpm <= 0:
      original_bpm = target_bpm

    original_bpm = np.atleast_1d(original_bpm)[0]  # ensures value is a single number

    tempo_multiplier = float(target_bpm) / float(original_bpm) #ratio to stretch or compress time
    #librosa.effects.time_stretch documentation: https://librosa.org/doc/main/generated/librosa.effects.time_stretch.html
    y = librosa.effects.time_stretch(y, rate=tempo_multiplier) #change speed


    #pitch/key matching
    #function inspiration: https://medium.com/@oluyaled/detecting-musical-key-from-audio-using-chroma-feature-in-python-72850c0ae4b1
    #librosa.feature.chroma_cqt documentation https://librosa.org/doc/main/generated/librosa.feature.chroma_cqt.html
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr) #extract pitch class features
    key_map = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    original_key = key_map[chroma.mean(axis=1).argmax()] #find most dominant pitch class
    n_steps = (key_map.index(Target_Keys[genre]) - key_map.index(original_key)) % 12 #calculate how many semitones to shift
    #librosa.effects.pitch_shift documentation: https://librosa.org/doc/main/generated/librosa.effects.pitch_shift.html
    if stem_name == "vocals":
      y = pitch_shifts(y=y, sr=sr, n_steps=n_steps, stem_type=stem_name)
    else:
      y = librosa.effects.pitch_shift(y=y, n_steps=n_steps, sr=sr)



    #apply genre eq
    y = apply_eq(y, sr, genre)

    # some stem specific mixing to enhance genre differences  #created by chatGPT
    if stem_name == "drums":
        if genre in ["EDM", "Techno", "Dubstep"]:
            y *= 1.8
        elif genre == "Lo-Fi":
            y *= 0.6

    if stem_name == "bass":
        if genre in ["EDM", "Dubstep"]:
            y *= 2.0
        elif genre == "Lo-Fi":
            y *= 0.7

    if stem_name == "vocals":
        if genre == "Lo-Fi":
            y = add_reverb(y, sr, 0.5)
        elif genre in ["Metal", "Grunge"]:
            y = add_distortion(y, 1.5)

    # add distortion to rock genres #created by chatGPT
    if genre in ["Rock", "Metal", "Grunge"]:
        y = add_distortion(y, 2.5)

    # genre specific reverb #created by chatGPT
    if genre == "Cinematic":
        y = add_reverb(y, sr, strength=0.8, decay=0.5)
    elif genre == "Lo-Fi":
        y = add_reverb(y, sr, strength=0.3, decay=0.2)
    elif genre == "EDM":
        y = add_reverb(y, sr, strength=0.1, decay=0.05)
    elif preset["reverb"] > 0:
        y = add_reverb(y, sr, preset["reverb"])


    #harmonic and percussive separation and boost
    #librosa.effects.hpss documentation: https://librosa.org/doc/main/generated/librosa.effects.hpss.html
    y_h, y_p = librosa.effects.hpss(y) #split audio into harmonic (melody) and percussive (rhythm)
    y = y_h + preset["percussive_boost"] * y_p #boost the rhythm based on genre by recombining harmonic and percussive (y harmonic + percussive boost * y percussive)

    #add reverb effect
    if preset["reverb"] > 0:
      y = add_reverb(y, sr, preset["reverb"])

    #normalize before saving
    #librosa.util.normalize documentation: https://librosa.org/doc/main/generated/librosa.util.normalize.html
    y = librosa.util.normalize(y)
    y = np.clip(y, -1.0, 1.0) #prevent clipping

    #save temp audio file for volume adjustments
    temp_file = f"temp_{os.path.basename(file_path)}"
    sf.write(temp_file, y, sr)

    #adjust volume using pydub
    audio = AudioSegment.from_wav(temp_file) #load into pydub for volume control
    audio += preset["gain"]

    #export remixed stem
    output_file = file_path.replace(".wav", "_remix.wav")
    audio.export(output_file, format="wav")

    return output_file


#Music Gen backing tracks (generates music and loops it to match song length)
def generate_musicgen(genre, target_duration, remix_audio, progress=gr.Progress()):
    progress(0.7, desc="Generating MusicGen AI Backing Track...")

    try:
        prompt = get_musicgen_prompt(genre) #retrieve the predetermined prompt
        inputs = processor(text=prompt,return_tensors="pt").to(device) #convert prompt into required tensor format

        with torch.no_grad():
            audio_values = musicgen_model.generate(**inputs, max_new_tokens=512) #max_new_tokens controls output length (more tokens = longer output)

        audio_array = audio_values[0].cpu().numpy() #extract generated waveform

        #ensure audio is mono so its usable
        if audio_array.ndim > 1:
          audio_array = audio_array[0]


        #save the generated short clip, convert to float32(standard audio format), use MusicGen default sample rate (32kHz)
        sf.write("musicgen_short.wav", audio_array.astype("float32"), 32000)

        #load generated audio into pydub
        base_audio = AudioSegment.from_file("musicgen_short.wav", format="wav")

        #tempo match MusicGen to remix
        remix_segment, sr = librosa.load(remix_audio, sr=None) #load remixed track
        # beat_track documentation: https://librosa.org/doc/main/generated/librosa.beat.beat_track.html
        remix_bpm, _ = librosa.beat.beat_track(y=remix_segment, sr=sr) #detect tempo of remix
        musicgen_bpm, _ = librosa.beat.beat_track(y=audio_array, sr=32000) #detect tempo of MusicGen audio

        #ensure tempos are not arrays to prevent errors
        #np.ndim returns number of dimensions in array
        remix_bpm = float(remix_bpm) if np.ndim(remix_bpm) > 0 else remix_bpm
        musicgen_bpm = float(musicgen_bpm) if np.ndim(musicgen_bpm) > 0 else musicgen_bpm


        #if tempo detection fails use remix tempo or default 120 bpm
        if musicgen_bpm <= 0:
          musicgen_bpm = remix_bpm if remix_bpm > 0 else 120

        tempo_ratio = remix_bpm / musicgen_bpm #calculate how much to stretch/compress to match remix tempo



        #librosa.effects.time_stretch documentation: https://librosa.org/doc/main/generated/librosa.effects.time_stretch.html
        stretched = librosa.effects.time_stretch(audio_array, rate=tempo_ratio) #adjust speed
        sf.write("musicgen_tempo.wav", stretched, 32000) #save tempo adjustment
        synced_audio = AudioSegment.from_file("musicgen_tempo.wav", format="wav") #load adjustment into pydub for looping

        #loop MusicGen clip with crossfade to match song duration. ensures MusicGen does not need to create a long track (creates slowness in system)
        looped = base_audio
        while len(looped) < target_duration:
          #pydub documentation: https://github.com/jiaaro/pydub
          looped = looped.append(base_audio, crossfade=600) #repeat audio until it matches target duration, crossfade helps smooth loop transitions


        full_audio = looped[:target_duration] #trim final loop to target duration
        full_audio.export("musicgen_full.wav", format="wav") #export backing track

        return "musicgen_full.wav" #return backing track file path

    #for MusicGen debugging
    except Exception as e:
      print("Music Gen error:", e)
      return None #system continues if MusicGen fails




def generate_spectrograms(original, remixed):

    y_orig, sr_orig = librosa.load(original, sr=None) #load original audio waveform and sample rate
    y_new, sr_new = librosa.load(remixed, sr=None) #load remixed audio waveform and sample rate

    ####Spectrograms comparison

    ##librosa melspectrogram documentation: https://librosa.org/doc/main/generated/librosa.feature.melspectrogram.html
    s_orig = librosa.feature.melspectrogram(y=y_orig, sr=sr_orig) #convert audio to frequency representation over time
    ##librosa.power_to_db documentation: https://librosa.org/doc/main/generated/librosa.power_to_db.html
    s_orig_db = librosa.power_to_db(s_orig, ref=np.max) #convert power to decibels
    s_new = librosa.feature.melspectrogram(y=y_new, sr=sr_new)
    s_new_db = librosa.power_to_db(s_new, ref=np.max)

    spectro_path = remixed.replace(".wav", "_spectrogram.png") #ouput file path

    plt.figure(figsize=(12, 6)) #create canvas
    ##matplotlib documentation: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html
    plt.subplot(2,1,1)  #original is top plot
    ##librosa.display.specshow documentation: https://librosa.org/doc/main/generated/librosa.display.specshow.html
    librosa.display.specshow(s_orig_db, sr=sr_orig, x_axis='time', y_axis='mel') #display spectrogram
    plt.title("Original Spectrogram")
    ##matplotlib colorbar documentation: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.colorbar.html
    plt.colorbar(format="%+2.0f dB")

    plt.subplot(2,1,2) #remix is bottom plot
    librosa.display.specshow(s_new_db, sr=sr_new, x_axis='time', y_axis='mel')
    plt.title("Remix Spectrogram")
    plt.colorbar(format="%+2.0f dB")

    plt.tight_layout() #prevents labels from overlapping
    plt.savefig(spectro_path)
    plt.close() #save and close plot



    #####Rhythm comparison
    # onset.onset_strength documentation:  https://librosa.org/doc/main/generated/librosa.onset.onset_strength.html
    onset_orig = librosa.onset.onset_strength(y=y_orig, sr=sr_orig) #measure rhythmic energy (where beats occur)
    onset_new = librosa.onset.onset_strength(y=y_new, sr=sr_new)
    rhythm_path = remixed.replace(".wav", "_rhythm.png")

    plt.figure(figsize=(12, 3))
    #compare rhythm intensity over time
    plt.plot(onset_orig, label="Original")
    plt.plot(onset_new, label="Remix")

    plt.title("Rhythm / Beat Intensity")
    plt.legend() #create legend (documentation: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html)
    plt.tight_layout() #prevents labels from overlapping
    plt.savefig(rhythm_path)
    plt.close()



    ####Pitch comparison
    chroma_orig = librosa.feature.chroma_cqt(y=y_orig, sr=sr_orig) #extract pitch class
    chroma_new = librosa.feature.chroma_cqt(y=y_new, sr=sr_new)
    pitch_path = remixed.replace(".wav", "_chroma.png")
    plt.figure(figsize=(12, 6))

    plt.subplot(2,1,1)
    librosa.display.specshow(chroma_orig, y_axis='chroma', x_axis='time')
    plt.title("Original Pitch (Chroma)")

    plt.subplot(2,1,2)
    librosa.display.specshow(chroma_new, y_axis='chroma', x_axis='time')
    plt.title("Remix Pitch (Chroma)")

    plt.tight_layout() #prevents labels from overlapping
    plt.savefig(pitch_path)
    plt.close()



    ###Energy comparison
    #librosa.feature.rms documentation: https://librosa.org/doc/main/generated/librosa.feature.rms.html
    rms_orig = librosa.feature.rms(y=y_orig)[0] #root mean square = perceived loudness
    rms_new = librosa.feature.rms(y=y_new)[0]
    energy_path = remixed.replace(".wav", "_energy.png")
    plt.figure(figsize=(12, 3))

    #compare loudness over time
    plt.plot(rms_orig, label="Original")
    plt.plot(rms_new, label="Remix")
    plt.title("Energy / Loudness (RMS)")
    plt.legend()
    plt.tight_layout() #prevents labels from overlapping
    plt.savefig(energy_path)
    plt.close()

    return spectro_path, rhythm_path, pitch_path, energy_path #return all vis file paths



#preview stems (returns stems for UI preview feature)
def preview_stems(audio_file, progress=gr.Progress()):
  if audio_file is None:
    return None, None, None, None, "Please Upload a File"

  audio_file = convert_to_wav(audio_file)
  stems = separate_stems(audio_file, progress)

  return (
      stems["vocals"],
      stems["drums"],
      stems["bass"],
      stems["other"],
      "Stems Ready"
  )



#####MAIN REMIXING PIPELINE
def dj_pal(audio_file, selected_stems, genre, use_musicgen, progress=gr.Progress()):

    if audio_file is None:
        return None, None, None, None, None, None, None, None, "Please Upload a File" #early exit if no uploaded file


    #ensure file is wav
    audio_file = convert_to_wav(audio_file)

    #Separate stems (vocals, bass, drums, other)
    stem_path = separate_stems(audio_file, progress)

    progress(0.5, desc="Remixing Your Selected Stems...")

    preset = Genre_Presets[genre] #load genre parameters
    stem_names = ["vocals", "drums", "bass", "other"]
    remixed_files = []


    for stem in stem_names:
        path = stem_path[stem]

        #skip missing stems
        if not os.path.exists(path):
          continue

        #only remixed selected stems
        if stem in selected_stems:
          processed = remix_stem(path, preset, genre, stem_name=stem) #apply remix transformation
        else:
          processed = path #keep original stem

        remixed_files.append(processed)


    if not remixed_files:
        return None, None, None, None, None, None, None, None, "No Stems Selected"

    #combine stems
    combined = AudioSegment.from_wav(remixed_files[0]) - 6 #start with first stem + reduce volume
    for f in remixed_files[1:]:
        combined = combined.overlay(AudioSegment.from_wav(f) - 6) #overlay additional stems + reduce volume

    #from pydub documentation: https://github.com/jiaaro/pydub/blob/master/API.markdown
    combined = combined.apply_gain(-combined.max_dBFS) #normalize loudness

    #optional music gen backing track
    if use_musicgen:
      backing_path = generate_musicgen(genre, len(combined), remixed_files[0], progress)

      if backing_path and os.path.exists(backing_path):
        backing_audio = AudioSegment.from_file(backing_path, format="wav")
        # backing_audio = normalize(backing_audio)
        combined = combined.overlay(backing_audio - 6)

      else:
        print("MusicGen failed")

    #export final result
    output_file = "final_remix.wav"
    combined.export(output_file, format="wav")


    spectrogram_img_path, rhythm_path, pitch_path, energy_path = generate_spectrograms(audio_file, output_file) #generate visuals for analysis

    return(
        output_file,
        output_file,
        stem_path["vocals"], stem_path["drums"], stem_path["bass"], stem_path["other"], #stems
        spectrogram_img_path,  # spectrogram_img
        rhythm_path,
        pitch_path,
        energy_path,
        "Remix Complete!"
    )



# User interface using Gradio
#gradio documentation: https://www.gradio.app/guides/quickstart
with gr.Blocks(theme=gr.themes.Soft()) as app:

    #title and sub title
    gr.Markdown("<h1>DJ Pal</h1>")
    gr.Markdown("<h2>A Generative Art System for Musical Element Separation and Remix Creation</h2>")

    #gradio column documentation: https://www.gradio.app/docs/gradio/column
    with gr.Row():
        with gr.Column(scale=2): #make the column twice as wide as the row
          audio_input = gr.Audio(type="filepath", label="Upload a Song")

          #preview stems before remix
          preview_btn = gr.Button("Preview Stems Before Remixing")
          preview_status = gr.Textbox(label="Preview Status")

          with gr.Row():
            vocals_audio = gr.Audio(label="Vocals", interactive=True)
            drums_audio = gr.Audio(label="Drums", interactive=True)
            bass_audio = gr.Audio(label="Bass", interactive=True)
            other_audio = gr.Audio(label="Other", interactive=True)



    preview_btn.click(
        preview_stems,
        inputs = [audio_input],
        outputs = [vocals_audio, drums_audio, bass_audio, other_audio, preview_status]
        )


    #remix controls
    with gr.Column(scale=1):
      selected_stems = gr.CheckboxGroup(
          ["vocals", "drums", "bass", "other"],
          label="Select Stems to Remix"
        )

      genre = gr.Dropdown(
          ["EDM", "Hip-Hop", "Lo-Fi", "Rock", "Industrial", "Punk", "Pop", "Metal", "Cinematic", "Grunge", "Dubstep", "Techno"],
          label="Select a Genre"
        )

      use_musicgen = gr.Checkbox(label="Add an AI Generated Genre Boost")

      generate_button = gr.Button("Generate Your Remix", variant="primary")
      status = gr.Textbox(label="Status")

    with gr.Row():
      with gr.Column(scale=2):
        output_audio = gr.Audio(label="Remix Preview")
        download_file = gr.File(label="Download Remix")

        #visuals section
        with gr.Column(scale=1):
          spectrogram_img_path = gr.Image(
          label="Spectrogram Analysis",
          height=600)
          rhythm_plot = gr.Image(label="Rhythm Analysis", height=600)
          pitch_plot = gr.Image(label="Pitch Analysis", height=600)
          energy_plot = gr.Image(label="Energy Analysis", height=600)



    generate_button.click(
        dj_pal,
        inputs=[audio_input, selected_stems, genre, use_musicgen],
        outputs=[
            output_audio, download_file,
            vocals_audio, drums_audio, bass_audio, other_audio,
            spectrogram_img_path, rhythm_plot, pitch_plot, energy_plot,
            status
        ]
    )


app.launch(share=True)

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1226/1010756216.py:570: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b334685b41db32a60.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
